# Multi-Agent Coordination using Tabular Q-Learning

## Objective

The objective of this project is to implement a multi-agent reinforcement learning system using Tabular Q-Learning.

Two autonomous agents (Type A and Type B) operate in the same 5×5 Grid World. Each agent learns independently to collect a sample and return it to its own base while avoiding collisions and adapting to the changing lake condition.

In [4]:
import random
import numpy as np
import matplotlib.pyplot as plt
import time

from IPython.display import clear_output

## Project Constants

In [5]:
# Grid Size
GRID_SIZE = 5

# Total Episodes
EPISODES = 20000

# Maximum Steps per Episode
MAX_STEPS = 100

## Q-Learning Hyperparameters

In [6]:
# Learning Rate
ALPHA = 0.1

# Discount Factor
GAMMA = 0.95

# Exploration Rate
EPSILON = 1.0

# Exploration Decay
EPSILON_DECAY = 0.995

# Minimum Exploration
MIN_EPSILON = 0.01

## Fixed Locations

In [7]:
# Landing Pad X (West)
X_ROW = 2
X_COL = 0

# Landing Pad Y (North)
Y_ROW = 0
Y_COL = 2

# Sampling Site U (East)
U_ROW = 2
U_COL = 4

# Sampling Site V (South)
V_ROW = 4
V_COL = 2

## Lake Configuration

In [8]:
# Lake Position (Center)

LAKE_ROW = 2
LAKE_COL = 2

# Lake State

DRY = 0
FLOODED = 1

# Initial Lake State

lake_state = DRY

# Probability that the lake changes state
LAKE_FLIP_PROBABILITY = 0.2

## Action Space

In [9]:
ACTIONS = [
    "UP",
    "DOWN",
    "LEFT",
    "RIGHT",
    "WAIT"
]

TOTAL_ACTIONS = len(ACTIONS)

print(TOTAL_ACTIONS)

5


## State Representation

In [10]:
TOTAL_STATES = (
    GRID_SIZE *
    GRID_SIZE *
    2 *
    2
)

print("Total States:", TOTAL_STATES)

Total States: 100


## State Encoding Function

In [11]:
def state_to_index(
    row,
    col,
    has_sample,
    lake_state
):

    return (
        (
            (
                row * GRID_SIZE
                + col
            )
            * 2
            + int(has_sample)
        )
        * 2
        + lake_state
    )

## Initialize Q-Tables

In [12]:
q_table_A = np.zeros(
    (
        TOTAL_STATES,
        TOTAL_ACTIONS
    )
)

q_table_B = np.zeros(
    (
        TOTAL_STATES,
        TOTAL_ACTIONS
    )
)

print("Q-Table A:", q_table_A.shape)
print("Q-Table B:", q_table_B.shape)

Q-Table A: (100, 5)
Q-Table B: (100, 5)


## Environment Reset Function

In [13]:
def reset_environment():
    """
    Reset the environment before every episode.
    """

    # Agent A starts from X
    agent_a_row = X_ROW
    agent_a_col = X_COL

    # Agent B starts from Y
    agent_b_row = Y_ROW
    agent_b_col = Y_COL

    # Initially no agent has a sample
    agent_a_has_sample = False
    agent_b_has_sample = False

    # Randomly initialize lake state
    lake_state = random.choice([DRY, FLOODED])

    return (
        agent_a_row,
        agent_a_col,
        agent_b_row,
        agent_b_col,
        agent_a_has_sample,
        agent_b_has_sample,
        lake_state
    )

## Test Environment Reset

In [14]:
(
    agent_a_row,
    agent_a_col,
    agent_b_row,
    agent_b_col,
    agent_a_has_sample,
    agent_b_has_sample,
    lake_state
) = reset_environment()

print("Agent A :", (agent_a_row, agent_a_col))
print("Agent B :", (agent_b_row, agent_b_col))

print("Agent A Sample :", agent_a_has_sample)
print("Agent B Sample :", agent_b_has_sample)

print("Lake State :", "DRY" if lake_state == DRY else "FLOODED")

Agent A : (2, 0)
Agent B : (0, 2)
Agent A Sample : False
Agent B Sample : False
Lake State : FLOODED


## Lake Flip Function

In [15]:
def update_lake_state(current_state):
    """
    Update lake state with probability p.
    """

    if random.random() < LAKE_FLIP_PROBABILITY:

        if current_state == DRY:
            return FLOODED

        return DRY

    return current_state

## Test Lake Flip

In [16]:
lake = DRY

for i in range(10):

    lake = update_lake_state(lake)

    print(
        "Step",
        i + 1,
        ":",
        "DRY" if lake == DRY else "FLOODED"
    )

Step 1 : DRY
Step 2 : DRY
Step 3 : DRY
Step 4 : DRY
Step 5 : DRY
Step 6 : FLOODED
Step 7 : FLOODED
Step 8 : DRY
Step 9 : DRY
Step 10 : DRY


## Pickup Logic

In [17]:
def pickup_sample(
    row,
    col,
    has_sample,
    target_row,
    target_col
):

    if (
        row == target_row
        and
        col == target_col
    ):
        has_sample = True

    return has_sample

## Delivery Logic

In [18]:
def deliver_sample(
    row,
    col,
    has_sample,
    home_row,
    home_col
):

    if (
        row == home_row
        and
        col == home_col
        and
        has_sample
    ):

        has_sample = False

        return True, has_sample

    return False, has_sample

## Environment Helper Functions

In [19]:
def is_lake_cell(row, col):

    return (
        row == LAKE_ROW
        and
        col == LAKE_COL
    )

## Test Helper Function

In [20]:
print(is_lake_cell(2, 2))
print(is_lake_cell(1, 1))

True
False


# Q-Learning Training

## Initialize Training Variables

In [21]:
# Store total reward of every episode
episode_rewards = []

# Statistics
collision_count = 0
water_damage_count = 0

delivery_count_A = 0
delivery_count_B = 0

## Start Training Loop

In [22]:
for episode in range(EPISODES):
    
    (
        agent_a_row,
        agent_a_col,
        agent_b_row,
        agent_b_col,
        agent_a_has_sample,
        agent_b_has_sample,
        lake_state
    ) = reset_environment()
    
    total_reward_A = 0
    total_reward_B = 0

## Get Current State

In [23]:
# Current State of Agent A

state_A = state_to_index(
    agent_a_row,
    agent_a_col,
    agent_a_has_sample,
    lake_state
)

# Current State of Agent B

state_B = state_to_index(
    agent_b_row,
    agent_b_col,
    agent_b_has_sample,
    lake_state
)

## Verify Current States

In [24]:
print("Agent A State Index:", state_A)
print("Agent B State Index:", state_B)

Agent A State Index: 41
Agent B State Index: 9


## Epsilon-Greedy Action Selection
## Select Action for Agent A

In [25]:
## Agent A choose an action

if random.random() < EPISODES:
    
    action_A = random.randint(0, TOTAL_ACTIONS -1 )
    
else:
    
    action_A = np.argmax(q_table_A[state_A])

## Select Action for Agent B

In [26]:
# Agent B chooses an action

if random.random() < EPISODES:
    
    action_B = random.randint(0, TOTAL_ACTIONS - 1)
    
else:
    
    action_B = np.argmax(q_table_B[state_B])

## Verify Selected Actions

In [27]:
print("Agent A Action:", ACTIONS[action_A])
print("Agent B Action:", ACTIONS[action_B])

Agent A Action: RIGHT
Agent B Action: UP


## Execute Joint Actions

In [28]:
# Next position of Agent A
next_a_row = agent_b_row
next_a_col = agent_b_col

# Next position of agent B
next_b_row = agent_b_row
next_b_col = agent_b_col

## Apply Actions

In [29]:
# ---------- Agent A ----------

if action_A == 0:
    next_a_row -= 1

elif action_A == 1:
    next_a_row += 1

elif action_A == 2:
    next_a_col -= 1

elif action_A == 3:
    next_a_col += 1

# WAIT
elif action_A == 4:
    pass


# ---------- Agent B ----------

if action_B == 0:
    next_b_row -= 1

elif action_B == 1:
    next_b_row += 1

elif action_B == 2:
    next_b_col -= 1

elif action_B == 3:
    next_b_col += 1

# WAIT
elif action_B == 4:
    pass

## Keep Agents Inside the Grid

In [30]:
next_a_row = max(0, min(next_a_row, GRID_SIZE - 1))
next_a_col = max(0, min(next_a_col, GRID_SIZE - 1))

next_b_row = max(0, min(next_b_row, GRID_SIZE - 1))
next_b_col = max(0, min(next_b_col, GRID_SIZE - 1))

## Update Agent Positions

In [31]:
agent_a_row = next_a_row
agent_a_col = next_a_col

agent_b_row = next_b_row
agent_b_col = next_b_col

## Reward System

In [32]:
# Initial reward for this step

reward_A = 0
reward_B = 0

## Step and Wait Rewards

In [33]:
# ---------- Agent A ----------

if action_A == 4:
    reward_A += -3
else:
    reward_A += -5


# ---------- Agent B ----------

if action_B == 4:
    reward_B += -3
else:
    reward_B += -5

## Pickup Reward

In [34]:
# ---------- Agent A ----------

if (
    agent_a_row == U_ROW
    and
    agent_a_col == U_COL
    and
    not agent_a_has_sample
):

    agent_a_has_sample = True

    reward_A += 10


# ---------- Agent B ----------

if (
    agent_b_row == V_ROW
    and
    agent_b_col == V_COL
    and
    not agent_b_has_sample
):

    agent_b_has_sample = True

    reward_B += 10

## Delivery Reward

In [35]:
# ---------- Agent A ----------

delivered_A, agent_a_has_sample = deliver_sample(
    agent_a_row,
    agent_a_col,
    agent_a_has_sample,
    X_ROW,
    X_COL
)

if delivered_A:

    reward_A += 50

    delivery_count_A += 1


# ---------- Agent B ----------

delivered_B, agent_b_has_sample = deliver_sample(
    agent_b_row,
    agent_b_col,
    agent_b_has_sample,
    Y_ROW,
    Y_COL
)

if delivered_B:

    reward_B += 50

    delivery_count_B += 1

## Update Episode Rewards

In [36]:
total_reward_A += reward_A
total_reward_B += reward_B

## Collision Detection

In [37]:
collision = False

if (
    next_a_row == next_b_row
    and
    next_a_col == next_b_col
    and
    next_a_row == LAKE_ROW
    and
    next_a_col == LAKE_COL
):

    collision = True

## Apply Collision Penalty

In [38]:
if collision:

    reward_A -= 20
    reward_B -= 20

    collision_count += 1

## Verify Collision Status

In [39]:
if collision:
    print("💥 Collision Detected!")

## Water Damage Logic

In [40]:
# Water damage applies only to Agent A

if (
    lake_state == FLOODED
    and
    next_a_row == LAKE_ROW
    and
    next_a_col == LAKE_COL
):

    reward_A -= 20

    water_damage_count += 1

# Agent B is waterproof.
# No water damage penalty is applied.

## Verify Water Damage

In [41]:
if (
    lake_state == FLOODED
    and
    next_a_row == LAKE_ROW
    and
    next_a_col == LAKE_COL
):
    print("🌊 Agent A received water damage!")

## Get Next State

In [45]:
# Next state of agent A
next_state_A = state_to_index(
    agent_a_row,
    agent_a_col,
    agent_a_has_sample,
    lake_state
)
# Next state of agent B
next_state_B = state_to_index(
    agent_b_row,
    agent_b_col,
    agent_b_has_sample,
    lake_state
)

## Verify Next State

In [46]:
print("Next State A :", next_state_A)
print("Next State B :", next_state_B)

Next State A : 13
Next State B : 9


## Q-Table Update

In [47]:
# ---------- Agent A Q-Update ----------

best_next_action_A = np.max(q_table_A[next_state_A])

q_table_A[state_A][action_A] = (
    q_table_A[state_A][action_A]
    +
    ALPHA
    * (
        reward_A
        +
        GAMMA * best_next_action_A
        -
        q_table_A[state_A][action_A]
    )
)


# ---------- Agent B Q-Update ----------

best_next_action_B = np.max(q_table_B[next_state_B])

q_table_B[state_B][action_B] = (
    q_table_B[state_B][action_B]
    +
    ALPHA
    * (
        reward_B
        +
        GAMMA * best_next_action_B
        -
        q_table_B[state_B][action_B]
    )
)

## Epsilon Decay

In [49]:
# Store total reward of this episode
episode_rewards.append(
        (total_reward_A + total_reward_B) / 2
    )

    # Decay epsilon
if EPSILON > MIN_EPSILON:
        EPSILON *= EPSILON_DECAY
        EPSILON = max(EPSILON, MIN_EPSILON)

## Training Complete

In [50]:
print("✅ Training Completed!")

print(f"Final Epsilon: {EPSILON:.4f}")
print(f"Total Collisions: {collision_count}")
print(f"Water Damage Count: {water_damage_count}")
print(f"Agent A Deliveries: {delivery_count_A}")
print(f"Agent B Deliveries: {delivery_count_B}")

✅ Training Completed!
Final Epsilon: 0.9900
Total Collisions: 0
Water Damage Count: 0
Agent A Deliveries: 0
Agent B Deliveries: 0
